In [ ]:
from pathlib import Path
import sys

# Localizar la raiz del repositorio desde Jupyter o terminal
RAIZ_REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "notebooks").is_dir() and (p / "datos").is_dir()),
    None
)
if RAIZ_REPO is None:
    raise FileNotFoundError("Ejecuta desde el repositorio caso-itam")

sys.path.insert(0, str(RAIZ_REPO / "src"))


## Estimadores

In [ ]:
# Carga y diagnóstico de poblaciones

# Objetivo: antes de ajustar el factor, ver qué tan lejos está la población
# que "estima" la ENVIPE de las dos referencias candidatas (CONAPO y Censo 2020).
# Nada se modifica todavía; solo medimos.

import numpy as np
import pandas as pd
from pathlib import Path

CARPETA = RAIZ_REPO / "datos" / "procesados"

inc         = pd.read_parquet(CARPETA / "envipe_incidentes.parquet")
personas    = pd.read_parquet(CARPETA / "envipe_personas.parquet")   # marco de UPM + FAC_ELE
maestro     = pd.read_parquet(CARPETA / "maestro_alcaldia.parquet")
directas_v1 = pd.read_parquet(CARPETA / "estimaciones_directas.parquet")

print(f"Incidentes: {len(inc):,} | personas: {len(personas):,} | "
      f"maestro: {maestro.shape} | directas v1: {directas_v1.shape}")

# --- 1. Población de 18 años y más que estima la encuesta ---
# Cada persona elegida (TPer_Vic2) representa FAC_ELE personas de 18+.
# Sumar FAC_ELE por alcaldía de residencia = población 18+ según la ENVIPE.
pob_envipe = (personas[personas["ENT_RES"] == "09"]
              .groupby(["anio_envipe", "MUN_RES"])["FAC_ELE"].sum()
              .rename("pob_envipe")
              .reset_index()
              .rename(columns={"MUN_RES": "cve_mun"}))

# La ENVIPE t pregunta por delitos del año t-1: ese es el anio_ref del maestro
pob_envipe["anio_ref"] = pob_envipe["anio_envipe"] - 1

# --- 2. Pegamos las dos referencias del maestro (ambas de 18+) ---
comp = pob_envipe.merge(
    maestro[["cve_mun", "nombre", "anio_ref", "pob_18_conapo", "pob_18_censo2020"]],
    on=["cve_mun", "anio_ref"], how="outer", validate="1:1")

# Control: 16 alcaldías x 5 años, sin huecos en ningún lado
assert len(comp) == 80, f"Esperaba 80 filas, hay {len(comp)}"
assert comp.notna().all().all(), "Hay alcaldía-año sin población en alguna fuente"

# El cociente de la ecuación (8): población de referencia / población estimada.
# Es el número por el que se multiplicaría FAC_DEL de los residentes de esa alcaldía.
comp["aj_conapo"] = comp["pob_18_conapo"]    / comp["pob_envipe"]
comp["aj_censo"]  = comp["pob_18_censo2020"] / comp["pob_envipe"]

# --- 3. Nivel CDMX: ¿el INEGI calibró a algo parecido a CONAPO o al censo? ---
cdmx = comp.groupby("anio_envipe")[["pob_envipe", "pob_18_conapo", "pob_18_censo2020"]].sum()
cdmx["aj_conapo"] = cdmx["pob_18_conapo"]    / cdmx["pob_envipe"]
cdmx["aj_censo"]  = cdmx["pob_18_censo2020"] / cdmx["pob_envipe"]
print("\n===== CDMX completa (millones de personas de 18+) =====")
print((cdmx[["pob_envipe", "pob_18_conapo", "pob_18_censo2020"]] / 1e6).round(3)
      .join(cdmx[["aj_conapo", "aj_censo"]].round(4)).to_string())

# --- 4. Nivel alcaldía: el cociente que de verdad se aplicaría ---
tabla = comp.pivot(index="nombre", columns="anio_envipe", values="aj_conapo")
print("\n===== Cociente CONAPO / ENVIPE por alcaldía (1 = sin ajuste) =====")
print(tabla.round(3).to_string())

for col in ["aj_conapo", "aj_censo"]:
    s = comp[col]
    print(f"\n{col}: mín {s.min():.3f} | mediana {s.median():.3f} | máx {s.max():.3f} | "
          f"celdas que se mueven más de 10%: {((s - 1).abs() > 0.10).sum()} de 80")

Decisión: CONAPO del año de referencia. 

Argumentos:

1. A nivel alcaldía, CONAPO y censo dan prácticamente la misma corrección (0.50–1.34 contra 0.49–1.34). Lo que los distingue es la tendencia, y el censo no la tiene.
2. CONAPO es el denominador de las tasas de C. Usar la misma población en la ENVIPE y en la Fiscalía es lo que hace comparable la correlación de rangos.
3. El desfase entre el levantamiento (marzo de t) y el año del hecho (t − 1) mueve la población medio punto porcentual. Es despreciable frente a correcciones de ±50%.

In [ ]:
# Ajuste del factor a la población CONAPO (ec. 8 de Santillana)

# Regla: el factor de cada persona/incidente se multiplica por
#        c_a = población CONAPO 18+ (año de referencia) / población 18+ que estima la ENVIPE
# donde "a" es la alcaldía de RESIDENCIA, porque el factor de expansión es una
# propiedad de la persona entrevistada (dónde vive), no de dónde le ocurrió el delito.

# Decisión documentada: referencia = CONAPO del anio_ref (misma que los denominadores de C).

# --- 1. Tabla de cocientes: una fila por alcaldía-año ---
ajuste = comp[["anio_envipe", "cve_mun", "aj_conapo"]].rename(columns={"aj_conapo": "c_aj"})

# --- 2. Personas: ajustamos FAC_ELE (lo necesitaremos para tasas) ---
# assign(ENT_RES="09") hace que solo peguen los residentes de la CDMX;
# un MUN_RES "002" de otro estado no recibe el cociente de Azcapotzalco.
personas = personas.drop(columns="c_aj", errors="ignore").merge(
    ajuste.rename(columns={"cve_mun": "MUN_RES"}).assign(ENT_RES="09"),
    on=["anio_envipe", "ENT_RES", "MUN_RES"], how="left", validate="m:1")
personas["c_aj"] = personas["c_aj"].fillna(1.0)          # fuera de la CDMX: sin ajuste
personas["FAC_ELE_AJ"] = personas["FAC_ELE"] * personas["c_aj"]

# --- 3. Incidentes: ajustamos FAC_DEL según la alcaldía de residencia de la víctima ---
# drop(errors="ignore") permite volver a correr la celda sin duplicar la columna
inc = inc.drop(columns="c_aj", errors="ignore").merge(
    ajuste.rename(columns={"cve_mun": "alc_res"}),
    on=["anio_envipe", "alc_res"], how="left", validate="m:1")
inc["c_aj"] = inc["c_aj"].fillna(1.0)                    # no residentes de la CDMX: factor 1
inc["FAC_DEL_AJ"] = inc["FAC_DEL"] * inc["c_aj"]

# --- 4. Controles ---
# (a) Por construcción, la población ajustada debe cuadrar EXACTO con CONAPO
chk = (personas[personas["ENT_RES"] == "09"]
       .groupby(["anio_envipe", "MUN_RES"])["FAC_ELE_AJ"].sum()
       .reset_index().rename(columns={"MUN_RES": "cve_mun"})
       .merge(comp, on=["anio_envipe", "cve_mun"], validate="1:1"))
assert len(chk) == 80 and np.allclose(chk["FAC_ELE_AJ"], chk["pob_18_conapo"]), "No cuadra con CONAPO"

# (b) Fuera de la CDMX nada debe cambiar (por eso las cifras nacionales no se moverán)
assert (personas.loc[personas["ENT_RES"] != "09", "c_aj"] == 1).all()
assert (inc.loc[~inc["cdmx_res"], "c_aj"] == 1).all()
print("Controles OK: población ajustada = CONAPO en las 80 celdas; fuera de la CDMX, factor intacto.")

# --- 5. ¿Cuánto puede mover el ajuste a d, k, R por OCURRENCIA? ---
# Si todos los incidentes de una alcaldía fueran de sus propios residentes, c_a se
# cancelaría en la razón y el ajuste no movería nada. Lo que sí mueve es la mezcla.
universo = ~inc["excl_vandalismo"] & inc["alc_ocu"].notna()   # mismo universo que el día 3
u = inc.loc[universo, ["alc_ocu", "alc_res", "cdmx_res", "FAC_DEL"]].copy()
u["origen"] = np.select([u["alc_res"] == u["alc_ocu"], u["cdmx_res"]],
                        ["misma alcaldía", "otra alcaldía"], default="fuera de la CDMX")

NOMBRE_ALC = maestro.drop_duplicates("cve_mun").set_index("cve_mun")["nombre"]
mezcla = (u.groupby(["alc_ocu", "origen"])["FAC_DEL"].sum()
           .unstack(fill_value=0))
mezcla = 100 * mezcla.div(mezcla.sum(axis=1), axis=0)          # % ponderado, cinco años juntos
mezcla.index = NOMBRE_ALC.reindex(mezcla.index)
print("\n===== ¿Dónde viven las víctimas de los delitos ocurridos en cada alcaldía? (%) =====")
print(mezcla[["misma alcaldía", "otra alcaldía", "fuera de la CDMX"]]
      .sort_values("misma alcaldía").round(1).to_string())

La comparación contra el INEGI se reporta con el factor original, que es lo que demuestra que reproducimos sus cifras. Las estimaciones que entran al modelo usan el factor ajustado. El ajuste a CONAPO mueve las proporciones de la CDMX menos de medio error estándar y reduce los totales entre 2% y 5%.

Una advertencia técnica para tenerla en mente: el error estándar trata a c_a como fijo, aunque en realidad depende de la muestra. **Santillana** hace lo mismo. En d, k y R no importa porque c_a casi se cancela; importaría si después modelas tasas.

In [ ]:
# Reverificación contra el INEGI: factor original vs. ajustado

# La rutina de Taylor (razon_rapida), con UN solo cambio:
# el argumento `peso`, para poder correrla con FAC_DEL o con FAC_DEL_AJ.
# Todo lo demás es idéntico (z, reindex a todas las UPM, n_h/(n_h-1)).

def razon_rapida(inc_a, upms, num, den, dominio, peso="FAC_DEL"):
    """Razón ponderada y su error estándar por linealización de Taylor.
    Σz² − (Σz)²/n reemplaza al .apply para que sea rápido."""
    w = inc_a[peso]
    y = w * inc_a[num] * dominio
    x = w * inc_a[den] * dominio
    if x.sum() == 0:                                   # sin denominador: no estimable
        return np.nan, np.nan
    razon = y.sum() / x.sum()
    z = (y - razon * x) / x.sum()                      # variable linealizada
    z_upm = (z.groupby([inc_a["EST_DIS"], inc_a["UPM_DIS"]]).sum()
              .reindex(upms, fill_value=0.0))          # UPM sin incidentes cuentan con z = 0
    por_estrato = z_upm.groupby(level="EST_DIS")
    n_h  = por_estrato.size()
    suma = por_estrato.sum()
    suma_cuadrados = (z_upm ** 2).groupby(level="EST_DIS").sum() - suma ** 2 / n_h
    varianza = (n_h / (n_h - 1) * suma_cuadrados).sum()
    return razon, np.sqrt(varianza)


ANIOS = sorted(inc["anio_envipe"].unique())
# Todas las UPM de cada año, con o sin incidentes (de la tabla de personas)
UPMS = {a: pd.MultiIndex.from_frame(
              personas.loc[personas["anio_envipe"] == a, ["EST_DIS", "UPM_DIS"]].drop_duplicates())
        for a in ANIOS}

# Los tres parámetros como (numerador, denominador). R se estima DIRECTO, no como d·k.
PARAMETROS = {"d": ("denuncio", "todos"),
              "k": ("carpeta",  "denuncio"),
              "R": ("carpeta",  "todos")}

DIA2 = {   # cifras del día 2 (factor original), ya verificadas contra el INEGI
    "d nacional": [10.07, 10.09, 10.90, 10.42, 9.58],
    "k nacional": [66.86, 67.29, 69.30, 67.99, 70.51],
    "d CDMX":     [11.20, 11.83, 11.09, 10.81, 9.97],
    "k CDMX":     [66.07, 65.92, 67.05, 64.64, 68.46],
}
PESOS = {"orig": ("FAC_DEL", "FAC_ELE"), "aj": ("FAC_DEL_AJ", "FAC_ELE_AJ")}

filas = []
for i, a in enumerate(ANIOS):
    inc_a = inc[inc["anio_envipe"] == a].assign(todos=True)
    per_a = personas[(personas["anio_envipe"] == a) & (personas["ENT_RES"] == "09")]
    sin_vand = ~inc_a["excl_vandalismo"]
    dominios = {"nacional": sin_vand,
                "CDMX":     sin_vand & inc_a["cdmx_ocu"]}      # ocurrencia, incluye alcaldía 999

    # (a) Proporciones con su error estándar
    for nombre_dom, dom in dominios.items():
        for param, (num, den) in PARAMETROS.items():
            fila = {"anio": a, "dominio": nombre_dom, "cifra": param}
            for etiqueta, (peso_inc, _) in PESOS.items():
                est, ee = razon_rapida(inc_a, UPMS[a], num, den, dom, peso=peso_inc)
                fila[etiqueta], fila[f"ee_{etiqueta}"] = 100 * est, 100 * ee
            fila["dia2"] = DIA2.get(f"{param} {nombre_dom}", [np.nan] * 5)[i]
            filas.append(fila)

    # (b) Totales de la CDMX: aquí el ajuste SÍ debe notarse
    dom = dominios["CDMX"]
    fila_tot  = {"anio": a, "dominio": "CDMX", "cifra": "delitos (miles)"}
    fila_tasa = {"anio": a, "dominio": "CDMX", "cifra": "tasa por 100 mil"}
    for etiqueta, (peso_inc, peso_per) in PESOS.items():
        total = (inc_a[peso_inc] * dom).sum()
        fila_tot[etiqueta]  = total / 1e3
        fila_tasa[etiqueta] = 1e5 * total / per_a[peso_per].sum()
    filas += [fila_tot, fila_tasa]

ver = pd.DataFrame(filas)
ver["dif"] = ver["aj"] - ver["orig"]                      # en puntos (o miles, o tasa)
ver["dif_en_ee"] = ver["dif"] / ver["ee_orig"]            # ¿cuántos errores estándar se movió?

# Control 1: con el factor ORIGINAL reproducimos el día 2 (y por tanto al INEGI)
con_ref = ver["dia2"].notna()
assert (ver.loc[con_ref, "orig"] - ver.loc[con_ref, "dia2"]).abs().max() < 0.006, "No reproduce el día 2"
print("Control OK: con FAC_DEL original se reproducen las 20 cifras del día 2.\n")

pd.set_option("display.width", 200)
for dom in ["nacional", "CDMX"]:
    print(f"===== {dom.upper()} =====")
    print(ver[ver["dominio"] == dom]
          .pivot(index="cifra", columns="anio", values=["orig", "aj", "dif", "dif_en_ee"])
          .round(2).to_string(), "\n")

prop = ver[ver["cifra"].isin(["d", "k", "R"])]
print(f"Movimiento máximo en proporciones: {prop['dif'].abs().max():.2f} puntos, "
      f"{prop['dif_en_ee'].abs().max():.2f} errores estándar")

Coincide al centésimo, así que el ajuste queda cerrado. Pasamos a la tabla completa.

Al construirla encontré algo que cambia el plan de la “primera prueba de consistencia”: R directo y d · k coinciden por identidad algebraica, no por evidencia. Ningún incidente tiene carpeta sin denuncia (lo verifiqué: cero casos), así que:

d̂ · k̂ = (Σ w·denunció / Σ w) · (Σ w·carpeta / Σ w·denunció) = Σ w·carpeta / Σ w = R̂

El denominador de k se cancela con el numerador de d. En las 480 filas la diferencia es de 10⁻¹⁷, es decir, cero de máquina. Como prueba no puede fallar.

R directo sigue siendo necesario, por otra razón: su error estándar no es el del producto. Su CV no se puede sacar de los de d y k sin la covarianza, y el modelo necesita esa varianza.

La prueba de consistencia con contenido aparece después de modelar. Tres modelos **Fay-Herriot** separados para d, k y R no están obligados a cumplir R = d · k, así que la distancia entre R̃ y d̃ · k̃ sí mide algo. 

In [ ]:
# Tabla completa de estimadores directos

# Para cada geografía × año × alcaldía × parámetro calculamos:
#   n       incidentes SIN ponderar en el denominador (todos para d y R; denunciados para k)
#   n_upm   UPM distintas que aportan al menos un incidente a ese denominador
#   est, ee estimación y error estándar (Taylor), con el factor AJUSTADO
#   cv      ee / est
#   deff    varianza de Taylor / varianza que tendría un muestreo aleatorio simple del mismo n
# También corremos el factor ORIGINAL, solo para verificar contra la v1.
import time

GEOGRAFIAS = {"ocurrencia": "alc_ocu", "residencia": "alc_res"}
CLAVES = sorted(maestro["cve_mun"].unique())               # las 16 alcaldías
PESOS_INC = {"orig": "FAC_DEL", "aj": "FAC_DEL_AJ"}

inicio = time.time()
filas = []
for a in ANIOS:
    inc_a = inc[inc["anio_envipe"] == a].assign(todos=True)
    sin_vand = ~inc_a["excl_vandalismo"]
    for geo, columna in GEOGRAFIAS.items():
        for clave in CLAVES:
            dominio = sin_vand & (inc_a[columna] == clave)
            for param, (num, den) in PARAMETROS.items():
                base = dominio & inc_a[den]                   # los incidentes del denominador
                fila = {"geografia": geo, "anio_envipe": a, "anio_ref": a - 1,
                        "alc": clave, "param": param,
                        "n": int(base.sum()),
                        "n_upm": inc_a.loc[base, ["EST_DIS", "UPM_DIS"]].drop_duplicates().shape[0]}
                for etiqueta, peso in PESOS_INC.items():
                    est, ee = razon_rapida(inc_a, UPMS[a], num, den, dominio, peso=peso)
                    fila[f"est_{etiqueta}"], fila[f"ee_{etiqueta}"] = est, ee
                # Kish: la parte del deff que viene solo de pesos desiguales (con el factor ajustado)
                w = inc_a.loc[base, "FAC_DEL_AJ"]
                fila["deff_kish"] = len(w) * (w ** 2).sum() / w.sum() ** 2 if len(w) > 0 else np.nan
                filas.append(fila)

directas = pd.DataFrame(filas)
print(f"{len(directas)} estimaciones en {time.time() - inicio:.0f} s")

# --- Columnas principales (factor ajustado) ---
directas["est"] = directas["est_aj"]
directas["ee"]  = directas["ee_aj"]
p, n = directas["est"], directas["n"]
interior = (p > 0) & (p < 1) & (n > 0)       # deff y CV solo tienen sentido lejos de 0 y 1
directas["cv"]   = np.where(p > 0, directas["ee"] / p, np.nan)
directas["deff"] = np.where(interior, directas["ee"] ** 2 / (p * (1 - p) / n), np.nan)
directas["n_efectiva"] = n / directas["deff"]
directas["nombre"] = directas["alc"].map(NOMBRE_ALC)

# --- Control 1: con el factor ORIGINAL debemos reproducir la v1 ---
v1 = directas_v1.rename(columns={"anio": "anio_envipe", "n": "n_v1",
                                 "est": "est_v1", "ee": "ee_v1"})
m = directas.merge(v1[["geografia", "anio_envipe", "alc", "param", "n_v1", "est_v1", "ee_v1"]],
                   on=["geografia", "anio_envipe", "alc", "param"], how="left", validate="1:1")
oc  = m[(m["geografia"] == "ocurrencia") & m["est_v1"].notna()]
res = m[(m["geografia"] == "residencia") & m["est_v1"].notna()]
assert np.allclose(oc["est_orig"], oc["est_v1"]) and np.allclose(oc["ee_orig"], oc["ee_v1"])
assert (oc["n"] == oc["n_v1"]).all()
assert np.allclose(res["est_orig"], res["est_v1"])
print("Control 1 OK: el factor original reproduce la v1 (ocurrencia: est, ee, n; residencia: est).")

# --- Control 2: por residencia el ajuste se cancela EXACTO (predicción de la celda 1) ---
r = directas[directas["geografia"] == "residencia"].dropna(subset=["est"])
print(f"Control 2: residencia, máx |ajustado − original| = {(r['est_aj'] - r['est_orig']).abs().max():.1e}")

# --- Control 3: R directo ≡ d·k (identidad algebraica, NO prueba empírica) ---
piv = directas.pivot_table(index=["geografia", "anio_envipe", "alc"], columns="param", values="est")
print(f"Control 3: máx |R − d·k| = {(piv['R'] - piv['d'] * piv['k']).abs().max():.1e}  (cero de máquina)")

# --- Cuánto movió el ajuste en ocurrencia ---
o = directas[directas["geografia"] == "ocurrencia"].copy()
o["mov_pp"] = 100 * (o["est_aj"] - o["est_orig"]).abs()
print(f"\nOcurrencia, movimiento por el ajuste (puntos): mediana {o['mov_pp'].median():.2f}, "
      f"máx {o['mov_pp'].max():.2f}")
print(o.nlargest(5, "mov_pp")[["anio_envipe", "nombre", "param", "n", "est_orig", "est_aj", "ee_aj", "mov_pp"]]
       .round(3).to_string(index=False))

# --- Celdas problemáticas: sin estimación o con EE = 0 (alimentan los puntos 5 y 6) ---
prob = directas[directas["est"].isna() | (directas["ee"] == 0)]
print(f"\nCeldas sin estimación o con EE = 0: {len(prob)}")
print(prob[["geografia", "anio_envipe", "nombre", "param", "n", "n_upm", "est", "ee"]]
      .sort_values(["geografia", "anio_envipe"]).to_string(index=False))

Un CV de cero no significa alta confiabilidad. Milpa Alta 2024 tiene k = 100% con 2 denuncias: su error estándar es 0 y su CV es 0%. Una clasificación mecánica la pondría como “alta confiabilidad”, que es exactamente lo contrario de la verdad. Por eso la celda pone dos categorías antes de aplicar umbrales:

sin estimación: no hay denominador.
degenerada: la estimación es 0 o 1, con error estándar cero.

Sobre **Buil-Gil**. Para un estimador directo, el sesgo es prácticamente nulo, así que su raíz del error cuadrático medio relativo es igual al CV. Con estimadores directos los dos criterios aplican umbrales distintos a la misma cantidad. La diferencia real aparece después de modelar, cuando el error cuadrático medio viene del modelo. Así lo dice la nota: aquí la comparación es de umbrales, no de métricas.

El CV castiga a las proporciones chicas y favorece a las grandes:

k vive cerca de 0.65; con un error estándar de 0.15, su CV es 23%.
Si reportaras la proporción complementaria (1 − k ≈ 0.35, la fracción de denuncias que no produce carpeta), el mismo error estándar daría un CV de 43%.

Por eso vemos a k “más confiable” que d, aunque tiene diez veces menos muestra. No es un error: es la escala. La celda imprime también el error estándar mediano para que lo veas en unidades absolutas, y en el escrito conviene una frase al respecto. 

In [ ]:
# Clasificación de confiabilidad (INEGI y Buil-Gil)

# INEGI (CV):            < 15% alta | 15–30% tolerable | > 30% con reservas
# Buil-Gil et al. (RRMSE): < 25% confiable | 25–50% con cautela | > 50% no usable
# Para un estimador directo (≈ insesgado), RRMSE = ee/est = CV: misma cantidad,
# distintos umbrales. Antes de aplicar umbrales separamos los casos que el CV no describe.

ORDEN_INEGI = ["alta", "tolerable", "con reservas", "degenerada", "sin estimación"]
ORDEN_BG    = ["confiable", "con cautela", "no usable", "degenerada", "sin estimación"]

cv_pct = 100 * directas["cv"]
sin_est    = directas["est"].isna()
degenerada = ~sin_est & ((directas["ee"] == 0) | (directas["est"] == 0))   # CV = 0 o indefinido
normal     = ~sin_est & ~degenerada

# np.select evalúa las condiciones en orden y toma la primera que se cumple
# default: etiqueta de texto para filas que no cumplan ninguna condición.
# Tiene que ser texto, no 0: las versiones nuevas de NumPy no mezclan números y cadenas.
directas["conf_inegi"] = np.select(
    [sin_est, degenerada, normal & (cv_pct < 15), normal & (cv_pct <= 30), normal],
    ["sin estimación", "degenerada", "alta", "tolerable", "con reservas"],
    default="SIN CLASIFICAR")
directas["conf_builgil"] = np.select(
    [sin_est, degenerada, normal & (cv_pct < 25), normal & (cv_pct <= 50), normal],
    ["sin estimación", "degenerada", "confiable", "con cautela", "no usable"],
    default="SIN CLASIFICAR")

# Las cinco condiciones cubren todos los casos; si algo cae en el default, hay un hueco lógico
assert not (directas[["conf_inegi", "conf_builgil"]] == "SIN CLASIFICAR").any().any(), \
    "Hay filas que no cayeron en ninguna categoría"

# Ordenamos las categorías para que las tablas salgan en orden lógico, no alfabético
directas["conf_inegi"]   = pd.Categorical(directas["conf_inegi"],   ORDEN_INEGI,  ordered=True)
directas["conf_builgil"] = pd.Categorical(directas["conf_builgil"], ORDEN_BG,     ordered=True)

# --- Tabla resumen para el escrito: celdas de 80 por geografía × parámetro ---
def resumen(columna):
    t = pd.crosstab([directas["geografia"], directas["param"]], directas[columna], dropna=False)
    t = t.reindex(columns=directas[columna].cat.categories, fill_value=0)
    assert (t.sum(axis=1) == 80).all(), "Cada fila debe sumar 80 celdas"
    return t

tabla_inegi   = resumen("conf_inegi")
tabla_builgil = resumen("conf_builgil")
print("===== Criterio INEGI (CV), celdas alcaldía-año de 80 =====")
print(tabla_inegi.to_string(), "\n")
print("===== Criterio Buil-Gil (RRMSE = CV en directos), celdas de 80 =====")
print(tabla_builgil.to_string(), "\n")

# --- Precisión en dos escalas: relativa (CV) y absoluta (EE en puntos) ---
print("===== Medianas por geografía y parámetro =====")
print(directas[normal]
      .groupby(["geografia", "param"])
      .agg(n_mediana=("n", "median"), est_mediana=("est", "median"),
           ee_mediana_pp=("ee", lambda s: 100 * s.median()),
           cv_mediano_pct=("cv", lambda s: 100 * s.median()))
      .round(1).to_string())

# --- Conciliación con las cifras del día 3 (d, ocurrencia) ---
o = directas[(directas["geografia"] == "ocurrencia") & (directas["param"] == "d")]
for etiqueta in ["orig", "aj"]:
    cv = (100 * o[f"ee_{etiqueta}"] / o[f"est_{etiqueta}"]).replace(np.inf, np.nan)
    print(f"\nd ocurrencia, factor {etiqueta:>4}: CV mediano {cv.median():.1f}% | "
          f"CV > 30%: {(cv > 30).sum()} de 80 ({100 * (cv > 30).sum() / 80:.1f}%)")

### Decisiones de diseño:

- Eje x logarítmico. n va de 2 a 631, y en escala lineal todas las celdas chicas, que son justo las que importan, quedarían apiladas contra el eje.
- Curva roja teórica. Es CV = √(deff · (1 − p) / (n · p)), con la mediana de p y del efecto de diseño de cada panel. Muestra que la nube de puntos no es ruido sin forma: sigue una ley. Con el tamaño de muestra que da la ENVIPE por alcaldía, ningún rediseño de cuestionario baja el CV de R del 30%. Ese es el argumento que convence al jurado.
- Franjas de color para el INEGI y líneas punteadas para **Buil-Gil**. Así quedan los dos antecedentes en una sola figura sin saturarla.
- Las celdas con error estándar cero, marcadas con ×. Si no se marcan, desaparecen del gráfico, porque su CV es cero o no está definido, y el lector creería que no existen.
- **Formato IEEE**. Mide 7.16 pulgadas de ancho, a dos columnas, y usa letra serif tamaño 8. Se guarda en PDF vectorial para el escrito y en PNG para la presentación.

In [ ]:
# Figura: CV contra tamaño de muestra (calidad de publicación)

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

CARPETA_FIG = RAIZ_REPO / "resultados" / "figuras"
CARPETA_FIG.mkdir(exist_ok=True)

# --- Estilo IEEE: serif, 8 pt, líneas finas; PDF con fuentes incrustadas (fonttype 42) ---
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "mathtext.fontset": "stix",                  # las letras matemáticas también en estilo Times
    "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 600, "pdf.fonttype": 42,
})

GEO_FIG = "ocurrencia"          # geografía principal; cambia a "residencia" para el apéndice
PANELES = [("d", r"(a) $d$: P(denuncia | delito)"),
           ("k", r"(b) $k$: P(carpeta | denuncia)"),
           ("R", r"(c) $R$: P(carpeta | delito)")]
TECHO = 90                      # el CV máximo es 83%; 90 deja aire arriba
COLOR_PUNTO, COLOR_CURVA = "#1f4e79", "#c0392b"

# sharey=True: los tres paneles comparten el eje de CV para compararlos a ojo;
# el eje x NO se comparte porque k vive en otra escala de n (2–88 contra 11–631).
fig, ejes = plt.subplots(1, 3, figsize=(7.16, 2.55), sharey=True,
                         gridspec_kw={"wspace": 0.08})

for ax, (param, titulo) in zip(ejes, PANELES):
    sub = directas[(directas["geografia"] == GEO_FIG) & (directas["param"] == param)]
    ok    = sub["cv"].notna() & (sub["ee"] > 0)          # CV bien definido
    deg   = sub["est"].notna() & ~ok                     # degeneradas: EE = 0
    nulas = sub["est"].isna()                            # sin estimación

    # Fondo: las tres zonas del INEGI; líneas sólidas INEGI, punteadas Buil-Gil
    ax.axhspan(0, 15,     color="#e8f3e8", zorder=0, lw=0)
    ax.axhspan(15, 30,    color="#fbf4e2", zorder=0, lw=0)
    ax.axhspan(30, TECHO, color="#f8e6e3", zorder=0, lw=0)
    for y in (15, 30):
        ax.axhline(y, color="0.25", lw=0.7, zorder=1)
    for y in (25, 50):
        ax.axhline(y, color="0.35", lw=0.6, ls=(0, (3, 2)), zorder=1)

    # Curva esperada: CV de una proporción con muestreo complejo = sqrt(deff·(1−p)/(n·p))
    p_med    = sub.loc[ok, "est"].median()
    deff_med = sub.loc[ok, "deff"].median()
    xs = np.geomspace(max(1, sub.loc[ok, "n"].min() * 0.7), sub["n"].max() * 1.4, 200)
    ax.plot(xs, 100 * np.sqrt(deff_med * (1 - p_med) / (xs * p_med)),
            color=COLOR_CURVA, lw=1.0, zorder=2)

    # Puntos: una alcaldía-año cada uno
    ax.scatter(sub.loc[ok, "n"], 100 * sub.loc[ok, "cv"], s=11, color=COLOR_PUNTO,
               alpha=0.75, edgecolor="white", linewidth=0.3, zorder=3)
    # Degeneradas en la base con ×; clip(lower=1) porque log(0) no existe
    if deg.any():
        ax.scatter(sub.loc[deg, "n"].clip(lower=1), np.full(deg.sum(), 2.0), marker="x",
                   s=16, color="black", linewidth=0.8, zorder=4, clip_on=False)

    ax.set_xscale("log")
    ax.set_ylim(0, TECHO)
    ax.set_title(titulo, loc="left", pad=4)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))   # 10, 100, 1000
    ax.tick_params(axis="x", which="minor", length=1.5)

    # Recuadro con el conteo que va al texto
    texto = f"CV > 30%: {(100 * sub.loc[ok, 'cv'] > 30).sum()} de 80"
    if deg.sum() or nulas.sum():
        texto += f"\nEE = 0: {deg.sum()}" + (f" · sin estimar: {nulas.sum()}" if nulas.sum() else "")
    ax.text(0.97, 0.96, texto, transform=ax.transAxes, ha="right", va="top", fontsize=6.8,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", lw=0.4))

# Rótulos de las zonas, solo en el panel (a), donde el lado izquierdo está vacío
for y, et in [(7.5, "alta"), (22.5, "tolerable"), (60, "con reservas")]:
    ejes[0].text(0.03, y, et, transform=ejes[0].get_yaxis_transform(), fontsize=6.5,
                 style="italic", color="0.35", va="center")

ejes[0].set_ylabel("Coeficiente de variación (%)")
fig.supxlabel("Incidentes en el denominador, sin ponderar (escala logarítmica)", fontsize=8, y=0.04)

leyenda = [
    Line2D([], [], marker="o", ls="", color=COLOR_PUNTO, ms=3.5, label="Alcaldía-año"),
    Line2D([], [], marker="x", ls="", color="black", ms=4, label="EE estimado = 0"),
    Line2D([], [], color=COLOR_CURVA, lw=1, label=r"CV esperado ($p$ y deff medianos)"),
    Line2D([], [], color="0.25", lw=0.7, label="INEGI: 15% y 30%"),
    Line2D([], [], color="0.35", lw=0.6, ls=(0, (3, 2)), label="Buil-Gil: 25% y 50%"),
]
fig.legend(handles=leyenda, loc="upper center", ncol=5, frameon=False,
           bbox_to_anchor=(0.5, 1.03), handletextpad=0.4, columnspacing=1.2)
fig.subplots_adjust(left=0.065, right=0.995, top=0.84, bottom=0.2)

for ext in ["pdf", "png"]:
    ruta = CARPETA_FIG / f"fig_cv_vs_n_{GEO_FIG}.{ext}"
    fig.savefig(ruta, bbox_inches="tight")
    print("Guardada:", ruta)
plt.show()

### Funciones generalizadas de varianza

Predicen la varianza a partir del tamaño de muestra en lugar de confiar en la estimación de cada celda.

**You–Hidiroglou** ajusta log(varianza) = a + b · log(n) con los datos. La pendiente y el intercepto salen de la regresión.

**Arcoseno** fija la pendiente en −1 por teoría y saca el intercepto del efecto de diseño: var(θ) = deff / (4n).

La diferencia de fondo está en la escala. En la escala de la proporción, la varianza depende de p(1 − p), y You–Hidiroglou no lo ve porque solo usa n. En la escala arcoseno, la varianza ya no depende de p, y por eso basta con n.

Para d y R, **You–Hidiroglou** se comporta como dice la teoría: pendiente cercana a −1 y R² ≈ 0.55. Para k, falla: la pendiente sale en −0.5 y el R² en 0.20. Justo donde está tu problema, el log-lineal no captura la varianza. La razón es que k varía mucho entre celdas (de 0.34 a 1) y la regresión ignora p.

**Arcoseno** se acerca más a la varianza directa en los seis grupos. La correlación con la varianza directa (en logaritmos) es de 0.72 a 0.88, contra 0.45 a 0.75 de You–Hidiroglou. Además, es más estrecha: el 80% central de los cocientes queda en 0.5–1.7, contra 0.55–2.8.
Las dos dan varianza definida en las celdas degeneradas. Ninguna resuelve Milpa Alta 2023 por ocurrencia, porque con n = 0 no hay nada que suavizar. Esa es materia de la celda 8.

In [ ]:
# Suavizamiento de la varianza: You–Hidiroglou vs. arcoseno

# Ambas rutas reemplazan la varianza directa (ruidosa o nula en celdas chicas)
# por una varianza PREDICHA a partir del tamaño de muestra.
#
# Ruta A, You–Hidiroglou (log-lineal):  log(v) = a + b·log(n) + error
#   Se ajusta por parámetro × geografía con las celdas de varianza bien definida.
#   Al regresar de logaritmos se multiplica por exp(σ²/2): si log(v) es normal,
#   E[v] = exp(media + σ²/2). Sin esa corrección, la varianza saldría subestimada.
#
# Ruta B, arcoseno:  θ = arcsen(√p),  var(θ) ≈ 1 / (4·n_ef),  n_ef = n / deff
#   La varianza de θ NO depende de p: está definida aunque p sea 0 o 1.
#   deff = mediana por parámetro × geografía (estable); el deff por celda es solo sensibilidad.

p, n = directas["est"], directas["n"]
interior = (p > 0) & (p < 1) & (n > 0) & (directas["ee"] > 0)    # varianza directa bien definida
directas["var_dir"] = directas["ee"] ** 2

# --- Ruta A: You–Hidiroglou ---
ajustes = []
for (geo, param), s in directas.groupby(["geografia", "param"]):
    usa = s.index[interior[s.index]]
    X = np.column_stack([np.ones(len(usa)), np.log(s.loc[usa, "n"])])   # columna de unos = intercepto
    y = np.log(s.loc[usa, "var_dir"])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)                         # mínimos cuadrados
    resid = y - X @ beta
    sigma2 = resid.var(ddof=2)                                           # ddof=2: estimamos 2 coeficientes
    con_n = s.index[s["n"] > 0]                                          # se puede predecir donde haya n
    directas.loc[con_n, "var_yh"] = np.exp(beta[0] + beta[1] * np.log(s.loc[con_n, "n"]) + sigma2 / 2)
    ajustes.append({"geografia": geo, "param": param, "celdas": len(usa),
                    "intercepto": beta[0], "pendiente": beta[1],
                    "R2": 1 - resid.var() / y.var(), "sigma2": sigma2})
ajustes = pd.DataFrame(ajustes)
print("===== Ruta A: ajuste log-lineal (la teoría predice pendiente ≈ −1) =====")
print(ajustes.round(3).to_string(index=False))

# --- Ruta B: arcoseno ---
directas["deff_suav"] = directas.groupby(["geografia", "param"])["deff"].transform("median")
directas["n_ef_suav"] = directas["n"] / directas["deff_suav"]
directas["theta"] = np.arcsin(np.sqrt(p))                                 # NaN si no hay estimación
directas["var_theta_asen"] = np.where(n > 0, 1 / (4 * directas["n_ef_suav"]), np.nan)
directas["var_theta_asen_celda"] = np.where(interior, directas["deff"] / (4 * n), np.nan)  # sensibilidad

# Varianza DIRECTA en escala θ (método delta: var(θ) ≈ var(p) / (4p(1−p))), para comparar
directas["var_theta_dir"] = np.where(interior, directas["var_dir"] / (4 * p * (1 - p)), np.nan)

# --- Comparación en las celdas donde existe la varianza directa ---
# Cada ruta se compara contra la directa EN SU PROPIA ESCALA.
filas = []
for (geo, param), s in directas[interior].groupby(["geografia", "param"]):
    r_yh = s["var_yh"] / s["var_dir"]
    r_as = s["var_theta_asen"] / s["var_theta_dir"]
    filas.append({
        "geografia": geo, "param": param,
        "YH: cociente mediano": r_yh.median(),
        "YH: p10–p90": f"{r_yh.quantile(.1):.2f}–{r_yh.quantile(.9):.2f}",
        "YH: corr log": np.corrcoef(np.log(s["var_yh"]), np.log(s["var_dir"]))[0, 1],
        "ASen: cociente mediano": r_as.median(),
        "ASen: p10–p90": f"{r_as.quantile(.1):.2f}–{r_as.quantile(.9):.2f}",
        "ASen: corr log": np.corrcoef(np.log(s["var_theta_asen"]), np.log(s["var_theta_dir"]))[0, 1],
    })
print("\n===== Suavizada / directa (1 = coinciden; p10–p90 estrecho = más estable) =====")
print(pd.DataFrame(filas).round(2).to_string(index=False))

# --- Lo que motiva todo: ¿qué varianza recibe cada celda problemática? ---
print("\n===== Celdas degeneradas o sin estimación: varianza que asigna cada ruta =====")
prob = directas[~interior].copy()
prob["ee_yh_pp"] = 100 * np.sqrt(prob["var_yh"])                  # EE en puntos porcentuales
prob["ee_theta_asen"] = np.sqrt(prob["var_theta_asen"])           # EE en radianes, escala θ
print(prob[["geografia", "anio_envipe", "nombre", "param", "n", "est",
            "ee_yh_pp", "ee_theta_asen"]].round(3).to_string(index=False))
print(f"\nCeldas con varianza suavizada definida — YH: {directas['var_yh'].notna().sum()} de 480 | "
      f"arcoseno: {directas['var_theta_asen'].notna().sum()} de 480")

El cociente mediano de 1.0 en arcoseno no es un mérito. Sale así por construcción, porque usé el deff mediano de cada grupo. Lo que sí habla a su favor son las otras dos columnas:

- Es más estrecho. El 80% central de los cocientes queda en 0.54–1.69, contra 0.55–2.82 de You–Hidiroglou.
- Correlaciona mejor con la varianza directa, sobre todo en k: 0.72 contra 0.45 por ocurrencia.

#### Decisión:

- Varianza principal: transformación arcoseno de la raíz, var(θ) = deff / (4n), con deff igual a la mediana por parámetro y geografía. 
- Sensibilidad: You–Hidiroglou log-lineal en log(n), con corrección exp(σ²/2), y arcoseno con el deff de cada celda.

Razones:

En k, el log-lineal no captura la varianza (pendiente −0.51 frente a −1 teórica, R² = 0.20), porque ignora la dependencia de p(1 − p). En la escala arcoseno esa dependencia desaparece.
Arcoseno reproduce mejor la varianza directa donde esta existe: correlación en logaritmos de 0.72–0.88, contra 0.45–0.75.
Da varianza definida en las celdas degeneradas sin depender de la estimación puntual.

Consecuencia: el **Fay-Herriot** se ajusta en la escala θ = arcsen(√p), con transformación de regreso.

In [ ]:
# Celdas sin estimación y degeneradas: inventario y tratamiento

# Tres tipos de celda, tres tratamientos en el modelo:
#   normal       → entra con su θ directo y su varianza suavizada
#   degenerada   → p = 0 o 1 con n > 0: ENTRA con θ = 0 o π/2 y varianza suavizada.
#                  Como su varianza es grande, el modelo le da poco peso y la jala
#                  hacia el sintético. No se descarta: "0 de 51" también es información.
#   sin estimación → n = 0: no hay θ. No entra al ajuste; recibe la predicción del modelo
#                  (sintético x'β en Fay-Herriot; x'β + efecto de la alcaldía estimado
#                  con los otros años en Rao-Yu). Es el análogo de los 1,347 municipios
#                  sin muestra en Santillana et al.

directas["tipo_celda"] = np.select(
    [directas["est"].isna(), ~interior, interior],
    ["sin estimación", "degenerada", "normal"], default="SIN CLASIFICAR")
assert not (directas["tipo_celda"] == "SIN CLASIFICAR").any()

TRATAMIENTO = {
    "normal":         "Entra al ajuste: θ directo, varianza arcoseno",
    "degenerada":     "Entra al ajuste: θ directo (0 o π/2), varianza arcoseno; el modelo la encoge al sintético",
    "sin estimación": "No entra al ajuste; recibe la predicción del modelo (sintético / Rao-Yu)",
}
directas["tratamiento_modelo"] = directas["tipo_celda"].map(TRATAMIENTO)

# --- 1. Conteo por geografía × parámetro ---
print("===== Tipo de celda, de 80 por geografía × parámetro =====")
print(pd.crosstab([directas["geografia"], directas["param"]], directas["tipo_celda"])
      .reindex(columns=["normal", "degenerada", "sin estimación"], fill_value=0).to_string())

# --- 2. Peso ilustrativo γ (NO es el modelo: momentos, sin covariables) ---
# En la escala θ, la variación observada entre celdas = variación real entre alcaldías (σ²_v)
# + ruido de muestreo (ψ). Despejando: σ²_v ≈ var(θ) − promedio(ψ).
# Centramos θ por año para no confundir el cambio en el tiempo con diferencias entre alcaldías.
# γ = σ²_v / (σ²_v + ψ) es la fracción del peso que el directo recibiría.
directas["gamma_ilustrativo"] = np.nan
resumen_gamma = []
for (geo, param), s in directas[directas["theta"].notna()].groupby(["geografia", "param"]):
    theta_c = s["theta"] - s.groupby("anio_envipe")["theta"].transform("mean")
    sigma2_v = max(0.0, theta_c.var() - s["var_theta_asen"].mean())
    gamma = sigma2_v / (sigma2_v + s["var_theta_asen"])
    directas.loc[s.index, "gamma_ilustrativo"] = gamma
    resumen_gamma.append({"geografia": geo, "param": param, "sigma2_v": sigma2_v,
                          "psi_mediana": s["var_theta_asen"].median(),
                          "gamma_mediana": gamma.median(), "gamma_min": gamma.min()})
print("\n===== Peso ilustrativo del estimador directo (γ), por momentos y sin covariables =====")
print(pd.DataFrame(resumen_gamma).round(4).to_string(index=False))

# --- 3. La lista que va a la nota ---
especiales = (directas[directas["tipo_celda"] != "normal"]
              .sort_values(["geografia", "param", "anio_envipe"])
              [["geografia", "anio_envipe", "anio_ref", "nombre", "param", "tipo_celda",
                "n", "n_upm", "est", "gamma_ilustrativo"]])
print(f"\n===== Celdas especiales: {len(especiales)} =====")
print(especiales.round(2).to_string(index=False))

La varianza entre alcaldías de k por residencia (0.0265) sale cuatro veces mayor que por ocurrencia (0.0068).

Mi primera hipótesis fue que la residencia mezcla fiscalías, porque incluye denuncias de delitos ocurridos en el Estado de México. Lo verifiqué y no se sostiene: solo el 5.9% de las denuncias de residentes de la CDMX corresponde a delitos ocurridos fuera de la ciudad. Lo más probable es ruido del estimador por momentos, que es inestable cuando la varianza entre alcaldías y el ruido de muestreo tienen el mismo orden. 

In [ ]:
# Guardar estimaciones_directas_v2.parquet y las tablas del día

# --- 1. Columnas de la v2, en orden lógico, con su descripción (diccionario) ---
DICCIONARIO_V2 = {
    # Llaves
    "geografia":   "Geografía del dominio: ocurrencia (principal) o residencia (robustez)",
    "anio_envipe": "Año del levantamiento ENVIPE",
    "anio_ref":    "Año de referencia = año del hecho = anio_envipe − 1 (LLAVE con el maestro)",
    "alc":         "Clave INEGI de la alcaldía, 3 dígitos (LLAVE con el maestro: cve_mun)",
    "nombre":      "Nombre de la alcaldía",
    "param":       "d = P(denuncia|delito); k = P(carpeta|denuncia); R = P(carpeta|delito), estimado directo",
    # Muestra
    "n":           "Incidentes sin ponderar en el denominador (todos para d y R; denunciados para k)",
    "n_upm":       "UPM que aportan al menos un incidente al denominador",
    # Estimador directo (factor ajustado a CONAPO)
    "est":         "Estimación directa (razón ponderada con FAC_DEL ajustado a CONAPO 18+ del anio_ref)",
    "ee":          "Error estándar por linealización de Taylor (estratos EST_DIS, UPM UPM_DIS)",
    "cv":          "Coeficiente de variación = ee/est (NaN si est = 0)",
    "deff":        "Efecto de diseño = ee² / [est(1−est)/n] (NaN si est ∈ {0,1})",
    "deff_kish":   "Parte del deff atribuible a pesos desiguales (Kish)",
    "n_efectiva":  "n / deff",
    "est_orig":    "Estimación con FAC_DEL original del INEGI (trazabilidad)",
    "ee_orig":     "Error estándar con FAC_DEL original del INEGI (trazabilidad)",
    # Confiabilidad
    "conf_inegi":   "INEGI: alta (<15%), tolerable (15–30%), con reservas (>30%), degenerada, sin estimación",
    "conf_builgil": "Buil-Gil: confiable (<25%), con cautela (25–50%), no usable (>50%), degenerada, sin estimación",
    "tipo_celda":   "normal | degenerada (est ∈ {0,1} o ee = 0, con n > 0) | sin estimación (n = 0)",
    "tratamiento_modelo": "Cómo entra la celda al modelo de áreas pequeñas",
    # Varianza suavizada — PRINCIPAL
    "theta":          "θ = arcsen(√est): variable respuesta del Fay-Herriot",
    "var_theta_asen": "PRINCIPAL: var(θ) = deff_suav / (4n)",
    "deff_suav":      "deff mediano por geografía × parámetro (usado en var_theta_asen)",
    "n_ef_suav":      "n / deff_suav",
    # Sensibilidad
    "var_theta_asen_celda": "SENSIBILIDAD: var(θ) con el deff de la propia celda",
    "var_yh":         "SENSIBILIDAD: varianza de est (escala p) suavizada You–Hidiroglou, log(v) ~ log(n)",
    "var_dir":        "Varianza directa de est (escala p) = ee²",
    "var_theta_dir":  "Varianza directa llevada a escala θ por método delta: var_dir / (4·est(1−est))",
}
# gamma_ilustrativo se queda FUERA del parquet a propósito: no es un estimador,
# y no queremos que en el día 9 alguien lo confunda con el γ del modelo.

faltan = [c for c in DICCIONARIO_V2 if c not in directas.columns]
assert not faltan, f"Faltan columnas en directas: {faltan}"

v2 = (directas[list(DICCIONARIO_V2)]
      .assign(conf_inegi=lambda t: t["conf_inegi"].astype(str),          # categorías → texto:
              conf_builgil=lambda t: t["conf_builgil"].astype(str))      # más portable entre versiones
      .sort_values(["geografia", "param", "anio_envipe", "alc"])
      .reset_index(drop=True))

# --- 2. Controles antes de guardar ---
assert len(v2) == 480, f"Esperaba 480 filas, hay {len(v2)}"
assert not v2.duplicated(["geografia", "anio_envipe", "alc", "param"]).any(), "Llave duplicada"
assert (v2.groupby(["geografia", "param"]).size() == 80).all()
assert v2["var_theta_asen"].isna().sum() == (v2["n"] == 0).sum(), "La varianza principal falta donde n > 0"
assert set(v2["anio_ref"]) == set(maestro["anio_ref"]), "Los años no pegan con el maestro"

# --- 3. Guardar y releer (controlamos sobre lo LEÍDO, como el día 2) ---
RUTA_V2 = CARPETA / "estimaciones_directas_v2.parquet"
v2.to_parquet(RUTA_V2, index=False)
v2_leido = pd.read_parquet(RUTA_V2)
pd.testing.assert_frame_equal(v2, v2_leido, check_dtype=False)
print(f"Guardado y verificado: {RUTA_V2.name} ({v2_leido.shape[0]} filas × {v2_leido.shape[1]} columnas)")

dicc_v2 = pd.DataFrame(DICCIONARIO_V2.items(), columns=["columna", "descripcion"])
dicc_v2.to_csv(CARPETA / "diccionario_estimaciones_v2.csv", index=False, encoding="utf-8-sig")

# --- 4. Excel con las tablas del día (para el escrito) ---
medianas = (v2[v2["tipo_celda"] == "normal"]
            .groupby(["geografia", "param"])
            .agg(n_mediana=("n", "median"), est_mediana=("est", "median"),
                 ee_mediana_pp=("ee", lambda s: 100 * s.median()),
                 cv_mediano_pct=("cv", lambda s: 100 * s.median())).round(2))

# La comparación de suavizamientos se recalcula aquí (no reusamos `filas` de la celda 7,
# porque ese nombre se ha sobrescrito varias veces en el cuaderno)
comparacion = []
for (geo, param), s in v2[v2["tipo_celda"] == "normal"].groupby(["geografia", "param"]):
    r_yh = s["var_yh"] / s["var_dir"]
    r_as = s["var_theta_asen"] / s["var_theta_dir"]
    comparacion.append({"geografia": geo, "param": param,
                        "YH_mediana": r_yh.median(), "YH_p10": r_yh.quantile(.1), "YH_p90": r_yh.quantile(.9),
                        "YH_corr_log": np.corrcoef(np.log(s["var_yh"]), np.log(s["var_dir"]))[0, 1],
                        "ASen_mediana": r_as.median(), "ASen_p10": r_as.quantile(.1), "ASen_p90": r_as.quantile(.9),
                        "ASen_corr_log": np.corrcoef(np.log(s["var_theta_asen"]), np.log(s["var_theta_dir"]))[0, 1]})
comparacion = pd.DataFrame(comparacion).round(3)

HOJAS = {
    "Confiabilidad_INEGI":    tabla_inegi.reset_index(),
    "Confiabilidad_BuilGil":  tabla_builgil.reset_index(),
    "Medianas_precision":     medianas.reset_index(),
    "Verificacion_INEGI":     ver.round(3),
    "Poblaciones_ajuste":     comp.round(4),
    "Ajuste_YouHidiroglou":   ajustes.round(4),
    "Comparacion_suaviz":     comparacion,
    "Celdas_especiales":      especiales.round(3),
    "Diccionario_v2":         dicc_v2,
}
RUTA_XLSX = CARPETA / "dia5_tablas.xlsx"
with pd.ExcelWriter(RUTA_XLSX) as escritor:          # "with" cierra el archivo aunque algo falle
    for hoja, tabla in HOJAS.items():
        tabla.to_excel(escritor, sheet_name=hoja, index=False)
print(f"Guardado: {RUTA_XLSX.name} con {len(HOJAS)} hojas: {', '.join(HOJAS)}")

# Nota de decisiones · Estimadores directos

## 1. Ajuste del factor de expansión (ec. 8 de Santillana et al., 2023)

**Decisión.** FAC_DEL y FAC_ELE se multiplican por c_a = población CONAPO 18+ del año de referencia ÷ población 18+ que estima la ENVIPE, según la alcaldía de **residencia** de la persona entrevistada. Las víctimas que no viven en la CDMX conservan el factor original (c = 1).

**Por qué CONAPO y no el censo.**
- (i) CONAPO es el denominador de las tasas de carpetas (`C`). Usar la misma población en la ENVIPE y en la Fiscalía hace comparables ambas fuentes.
- (ii) El censo está fijo en marzo de 2020 y no capta el cambio poblacional de 2021–2024.
- (iii) A nivel alcaldía, las dos referencias dan prácticamente la misma corrección (0.50–1.34 contra 0.49–1.34).
- El desfase entre el levantamiento (marzo de *t*) y el año del hecho (*t* − 1) es de alrededor de 0.5% de la población, despreciable frente a la magnitud de la corrección.

**Qué encontramos.**
- **El INEGI no calibra la ENVIPE ni contra CONAPO ni contra el censo.** En la CDMX, la población de 18+ que estima la encuesta crece de 7.35 a 7.76 millones (2021–2025); CONAPO, de 7.20 a 7.30. El cociente de la CDMX baja de 0.98 a 0.94. Pendiente: confirmar la población de calibración en la nota metodológica de cada levantamiento.
- **A nivel alcaldía la encuesta no está calibrada.** Los cocientes van de 0.50 a 1.34 y cambian de un año a otro sin patrón. En 51 de las 80 celdas la corrección supera ±10%. Es ruido de muestreo en la población expandida.

**Efecto del ajuste.**
- **Proporciones por residencia:** el ajuste se cancela exactamente (diferencia máxima de 3 × 10⁻¹⁶).
- **Proporciones por ocurrencia:** solo cambian a través de las víctimas que viven en otra alcaldía. Movimiento mediano de 0.27 puntos; el máximo es de 6.1 puntos (`k` de Tláhuac 2022, con n = 10 y error estándar de 21 puntos).
- **Agregado CDMX:** máximo de 0.81 puntos (≤ 0.43 errores estándar). El ajuste cambia el peso relativo de cada alcaldía dentro de la ciudad.
- **Agregado nacional:** máximo de 0.11 puntos (≤ 0.14 errores estándar).
- **Totales:** los delitos de la CDMX bajan entre 1.6% y 4.9%; la tasa por 100 mil se mueve menos de 2%.

**Verificación.** Con el factor original se reproducen exactamente las 20 cifras del día 2 (nacional y CDMX, `d` y `k`, 2021–2025). La comparación con el INEGI se reporta con el factor original; las estimaciones que entran al modelo usan el ajustado.

**Limitaciones declaradas.**
- (i) El error estándar trata a c_a como fijo, igual que Santillana. En `d`, `k` y `R` es irrelevante porque c_a casi se cancela; importaría si se modelaran tasas.
- (ii) Las víctimas no residentes, sin ajuste, representan entre 0.9% (Xochimilco) y 25.4% (Cuauhtémoc) de los incidentes por ocurrencia.

## 2. Tabla de estimadores directos

- **Archivo:** `estimaciones_directas_v2.parquet`, con 480 filas (2 geografías × 3 parámetros × 16 alcaldías × 5 años) y 28 columnas. Su diccionario está en `diccionario_estimaciones_v2.csv`.
- **Varianza:** linealización de Taylor con la rutina del día 3, validada contra `samplics`. El factor original reproduce la v1 exactamente (estimación, error estándar y `n`).
- **`R` se estima en forma directa.** Pero `R̂ = d̂ · k̂` es una **identidad algebraica**: ningún incidente tiene carpeta sin denuncia, así que el denominador de `k` se cancela con el numerador de `d` (diferencia máxima de 3 × 10⁻¹⁷). **No sirve como prueba de consistencia.** La prueba con contenido se traslada al día 9: comparar R̃ contra d̃ · k̃ entre modelos ajustados por separado, que no están obligados a cumplir la identidad. `R` directo se conserva por su varianza propia, que no se puede obtener de las de `d` y `k` sin su covarianza.

## 3. Confiabilidad (celdas alcaldía-año, de 80)

| Geografía | Parám. | Alta (<15%) | Tolerable | Con reservas (>30%) | Degenerada | Sin estimación |
|---|---|---|---|---|---|---|
| Ocurrencia | d | 0 | 47 | 32 | 1 | 0 |
| Ocurrencia | k | 31 | 37 | 9 | 2 | 1 |
| Ocurrencia | R | 0 | 28 | 51 | 1 | 0 |
| Residencia | d | 2 | 51 | 27 | 0 | 0 |
| Residencia | k | 31 | 37 | 8 | 4 | 0 |
| Residencia | R | 0 | 26 | 53 | 1 | 0 |

- **Buil-Gil.** En estimadores directos, la raíz del error cuadrático medio relativo es igual al CV, así que solo cambian los umbrales. Por ocurrencia, las celdas "no usables" (>50%) son 4 para `d`, 0 para `k` y 11 para `R`.
- **Las celdas degeneradas forman una categoría aparte.** Su CV es cero o no está definido, y clasificarlas por umbral las pondría como "alta confiabilidad".
- **Advertencia de escala.** El CV favorece a las proporciones grandes. `k` (mediana de 0.7) parece más confiable que `d` (0.1), aunque tiene una mediana de 23 denuncias contra 172 incidentes. En unidades absolutas, el error estándar mediano de `k` es de **11.7 puntos**, cuatro veces el de `d` (3.0).
- **CV medianos por ocurrencia:** `d` 27.8%, `k` 17.8%, `R` 33.2%. `R`, el índice de visibilidad, es el parámetro peor medido.
- **Conciliación con el día 3.** Para `d` por ocurrencia, el factor original da un CV mediano de 27.2% y 30 de 80 celdas (37.5%) por encima de 30%. El factor ajustado da 27.8% y 32 de 80 (40.0%). El escrito reporta las ajustadas y pone las originales en nota al pie.

**Figura.** `fig_cv_vs_n_ocurrencia.pdf` muestra el CV contra el tamaño de muestra, un panel por parámetro, con los umbrales del INEGI y de Buil-Gil y la curva esperada CV = √(deff (1 − p) / (n p)). La versión de residencia va al apéndice.

## 4. Suavizamiento de la varianza

**Decisión.**
- **Principal:** transformación arcoseno, θ = arcsen(√p) y var(θ) = deff / (4n), con deff igual a la mediana por geografía y parámetro (≈ 1.4–1.6).
- **Sensibilidad:** (i) You–Hidiroglou log-lineal, log(v) = a + b · log(n), con corrección exp(σ²/2); (ii) arcoseno con el deff de cada celda.

**Razones.**
1. El log-lineal funciona para `d` y `R` (pendientes de −0.96 a −1.08, R² ≈ 0.5–0.57), pero falla en `k` (pendiente de −0.51 y −0.57, R² de 0.20 y 0.25). Ignora la dependencia de p(1 − p), que en `k` es grande. En la escala arcoseno esa dependencia desaparece.
2. Arcoseno reproduce mejor la varianza directa donde esta existe: correlación en logaritmos de 0.72–0.88, contra 0.45–0.75, y cocientes más estrechos (el 80% central queda en 0.54–1.69, contra 0.55–2.82). Su cociente mediano de 1.0 se da por construcción y no se cuenta como evidencia.
3. Da varianza definida en las celdas degeneradas sin depender de la estimación puntual.
4. Se usa el deff mediano y no el de cada celda, porque este último no está definido en las degeneradas y es inestable con pocas denuncias.

**Consecuencia para el día 9.** El Fay-Herriot y el Rao-Yu se ajustan en la escala θ, con transformación de regreso y corrección de sesgo.

## 5. Celdas sin estimación y degeneradas

- **Ocurrencia:** 1 sin estimación y 4 degeneradas.
- **Residencia:** 5 degeneradas.

| Celda | Tipo | Tratamiento |
|---|---|---|
| Milpa Alta 2023 (ref. 2022), `k`, ocurrencia | Sin estimación (0 denuncias) | No entra al ajuste; recibe la predicción del modelo |
| Milpa Alta 2023, `d` y `R`, ocurrencia | Degenerada (0 de 51) | Entra con θ = 0 y varianza arcoseno |
| Milpa Alta 2024, `k`, ocurrencia | Degenerada (2 de 2) | Entra con θ = π/2 y varianza arcoseno |
| La Magdalena Contreras 2025, `k`, ocurrencia | Degenerada (5 de 5) | Entra con θ = π/2 y varianza arcoseno |
| Residencia: Miguel Hidalgo 2021 `k`; Milpa Alta 2023 `k` y `R`, 2024 `k`; La Magdalena Contreras 2025 `k` | Degeneradas | Mismo tratamiento |

- **Celda sin estimación.** En un Fay-Herriot de corte transversal recibe el sintético x'β̂. En el Rao-Yu recibe además el efecto de la alcaldía estimado con los otros cuatro años. Es el mismo tratamiento que Santillana et al. dan a los 1,347 municipios sin muestra.
- **Celdas degeneradas.** No se descartan, porque "0 de 51" también es información. Su varianza grande hace que el modelo las encoja hacia el sintético.
- **Peso ilustrativo** (momentos, sin covariables, *no es el modelo*). El γ mediano es 0.41 para `R`, 0.44 para `d` y 0.28 para `k` por ocurrencia. Las celdas degeneradas de `k` quedan en γ ≈ 0.04–0.08, es decir, más del 90% del peso iría al sintético.
- **Pendiente.** La varianza entre alcaldías de `k` por residencia sale cuatro veces la de ocurrencia. No es la mezcla de fiscalías: solo 5.9% de las denuncias de residentes ocurrieron fuera de la CDMX. Probablemente es inestabilidad del estimador por momentos; se reevalúa con el modelo.

## 6. Pendientes para el día 9

- [ ] Elegir la variante principal de `C` y declarar las otras tres como sensibilidad.
- [ ] No meter juntas `z_prop_fem_18` y `z_eje_socio_1` (correlación de −0.90).
- [ ] Recalcular la correlación de rangos entre la tasa de victimización y la tasa de carpetas con las estimaciones modeladas (antes: 0.08–0.43).
- [ ] Prueba de consistencia R̃ contra d̃ · k̃ entre modelos.
- [ ] Transformación de regreso de θ a p con corrección de sesgo.
- [ ] Confirmar la población de calibración del INEGI en la nota metodológica de la ENVIPE.